# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.3b - Import Pre-trained Model from S3

This notebook is an **alternative to notebook 3 (fine-tuning)**. Instead of running a full RLAIF training job (which can take a long time), you can use a pre-trained model that is already available on S3.

This notebook will:
1. Import the model artifacts from S3
2. Create a Model Package Group (if it doesn't exist)
3. Register the model as a Model Package in the SageMaker Model Registry

After running this notebook, you can proceed directly to **notebook 4 (evaluation)** and **notebook 5 (deployment)**.

---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
region = sess.boto_region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {region}")

project_prefix = "humanlike-rlaif"
base_model_jumpstart_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_shortname = "qwen25-7b"

## Download pre-trained model and upload to your S3 bucket

Download the model artifacts from the workshop asset URL and upload them to your own S3 bucket.

In [ ]:
pretrained_model_s3_uri = f"s3://{bucket_name}/{project_prefix}-{base_model_shortname}/checkpoints/hf_merged/"

!pip install -qU huggingface_hub 2>/dev/null

import os, gc
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_WORKERS"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# Free up memory before starting
gc.collect()

from huggingface_hub import snapshot_download
local_dir = "/tmp/qwen25-7b-hla"
snapshot_download(
    repo_id="esterk/qwen2.5-7b-hla",
    local_dir=local_dir,
    max_workers=1,
    resume_download=True,  # resume if interrupted
    etag_timeout=30,
)

!aws s3 sync {local_dir} {pretrained_model_s3_uri} --quiet
!rm -rf {local_dir}

print(f"\nPre-trained model S3 URI: {pretrained_model_s3_uri}")

### Verify the model artifacts exist on S3

In [ ]:
s3_client = boto3.client("s3")

# Parse bucket and prefix from the S3 URI
s3_parts = pretrained_model_s3_uri.replace("s3://", "").split("/", 1)
s3_bucket = s3_parts[0]
s3_prefix = s3_parts[1] if len(s3_parts) > 1 else ""

response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix, MaxKeys=10)

if "Contents" in response:
    print(f"✅ Found {response['KeyCount']} objects (showing up to 10):")
    for obj in response["Contents"]:
        print(f"  - {obj['Key']} ({obj['Size']:,} bytes)")
else:
    print("❌ No objects found at the specified S3 path. Please verify the URI.")

---

## Create Model Package Group

Create or retrieve the Model Package Group in the SageMaker Model Registry. This is the same group used by the fine-tuning notebook, so the evaluation and deployment notebooks will work seamlessly.

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name = f"{project_prefix}-{base_model_shortname}"

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless RLAIF customization",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

print(f"Model Package Group ARN: {model_package_group.model_package_group_arn}")

---

## Register the Model Package

Register the pre-trained model from S3 as a versioned Model Package. This makes it available for the evaluation and deployment notebooks, just as if it had been produced by a training job.

In [ ]:
sm_client = boto3.client("sagemaker", region_name=region)

response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="Pre-trained RLAIF model imported from S3",
    InferenceSpecification={
        "Containers": [
            {
                "ModelDataSource": {
                    "S3DataSource": {
                        "S3Uri": pretrained_model_s3_uri,
                        "S3DataType": "S3Prefix",
                        "CompressionType": "None",
                    }
                },
                "BaseModel": {
                    "HubContentName": base_model_jumpstart_id,
                    "HubContentVersion": "1.26.0",
                    "RecipeName": "verl-grpo-rlaif-qwen-2-dot-5-7b-instruct-lora",
                },
            }
        ],
    },
    ModelApprovalStatus="Approved",
)

model_package_arn = response["ModelPackageArn"]
print(f"✅ Created Model Package: {model_package_arn}")

### Verify the registered Model Package

In [ ]:
from sagemaker.core.resources import ModelPackage

model_package = ModelPackage.get(model_package_arn)

print(f"Model Package ARN: {model_package_arn}")
print(f"Model Package Group: {model_package_group_name}")
print(f"Status: {model_package.model_approval_status}")
print(f"S3 URI: {model_package.inference_specification.containers[0].model_data_source.s3_data_source.s3_uri}")
print(f"\n✅ Model is registered and ready. You can now proceed to:")
print(f"   - Notebook 4 (evaluation): 4-evaluation.ipynb")
print(f"   - Notebook 5 (deployment): 5-deployment.ipynb")